In [1]:
#backtest
import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

# Define the backtest time range (last 10 years)
end_date = datetime.now()
start_date = datetime(end_date.year-10, end_date.month, end_date.day)


In [2]:
# 1. Get the list of S&P 500 tickers (from Wikipedia or an index provider).
# For simplicity, here we use the yfinance Ticker for the S&P 500 index itself (SPY ETF) 
# and equal-weight index (RSP ETF) as proxies. Alternatively, one could scrape Wikipedia 
# to get all 500 tickers.
index_ticker = "SPY"      # S&P 500 ETF (market-cap weighted)
equal_ticker = "RSP"      # S&P 500 Equal Weight ETF for reference
# If you want individual stocks, you could fetch tickers via Wikipedia:
# tickers_df = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
# all_tickers = tickers_df['Symbol'].tolist()

In [3]:
# 2. Download historical monthly price data for SPY (and optionally RSP or individual stocks).
prices = yf.download([index_ticker, equal_ticker], start=start_date, end=end_date, interval="1mo", auto_adjust=True)['Close']

[*********************100%***********************]  2 of 2 completed


In [4]:
# 3. Fetch risk-free rate (3-month T-Bill) from FRED for the same period.
# We'll use pandas_datareader for FRED data (monthly frequency).
try:
    rf_rate = web.DataReader("TB3MS", "fred", start_date, end_date)  # 3-month T-bill monthly rate (%)
except Exception as e:
    rf_rate = None
# If rf_rate is None (e.g., no internet), we'll assume a constant 2% risk-free rate for demonstration.

Explanation: We installed necessary packages and imported them. We set the analysis period to the last 10 years up to today. For demonstration purposes, we use SPY (an ETF tracking the S&P 500) as a proxy for the market-cap weighted index and RSP (an ETF for the equal-weighted S&P 500) as a proxy for an equal-weight portfolio. (In a real scenario, one might retrieve all 500 individual stock tickers from Wikipedia and download each stock’s data. This is more data-intensive, so using index ETFs can simplify the illustration.) We attempt to fetch the 3-month Treasury Bill rate from FRED as TB3MS. If that fails (for example, if this environment has no internet access), we will fall back to assuming a fixed 2% risk-free rate for Sharpe ratio calculations.

Portfolio Backtest Implementation

With the price data in hand, we next compute monthly returns for the assets and implement the three portfolio strategies. All portfolios start with $100,000,000 and are rebalanced monthly:

	•	S&P 500 Index (Cap-Weighted): We simulate the performance of a market-cap weighted index. This can be done by directly using SPY’s returns (since SPY already represents the cap-weighted index with dividends). If we were using individual stocks, we would assign weights proportional to each stock’s market cap and let them float (the weight of each stock automatically changes as prices change, reflecting a buy-and-hold cap-weighted index ￼). For simplicity, using SPY’s price directly yields the index performance.
    
	•	Equal-Weighted Portfolio: We rebalance to equal weights every month. That means at the start of each month, each of the 500 stocks (or each stock in our proxy set) has a 0.2% weight (1/500). The monthly return of an equal-weighted portfolio is essentially the average of all constituent stocks’ returns in that month (since each stock contributes equally). We will use RSP (equal-weight S&P 500 ETF) as a reference, or compute it from individual stock returns if available.
    
	•	Momentum Portfolio (12-month momentum): At each monthly rebalance, we look back at the past 12 months of returns for each stock and select the top performers. We will choose, for example, the top 50 stocks (approximately top 10% of S&P 500) based on their 12-month total return (price appreciation over the last year) as our momentum winners. We invest equally in those winners for the next month. Every month, we repeat this process (drop stocks that fall out of the top 12-month performance group and add new winners). This is a cross-sectional momentum strategy (long the recent winners) ￼. Note: The first 12 months of the dataset are used to establish the initial momentum ranking (no trades in the first year since we need a 1-year lookback).

In [5]:
# Set initial capital
initial_capital = 100_000_000  # 100 million

# Compute monthly returns for SPY and RSP (and individual stocks if we had them)
ret_index = prices[index_ticker].pct_change().dropna()  # SPY returns
ret_equal_ref = prices[equal_ticker].pct_change().dropna()  # RSP returns (if needed for reference)

In [6]:
# If using individual stock data, compute their returns similarly:
# stock_prices = yf.download(all_tickers, start=start_date, end=end_date, interval="1mo", auto_adjust=True)['Adj Close']
# stock_returns = stock_prices.pct_change().dropna()

# For demo purposes, let's assume we have `stock_returns` for all S&P 500 constituents.
# Since we cannot fetch all here, we'll simulate a smaller universe using SPY and RSP for illustration.
stock_returns = pd.DataFrame({
    'SPY': ret_index,
    'RSP': ret_equal_ref
}).dropna()

In [7]:
# S&P 500 Index portfolio value (using SPY as proxy)
index_value = [initial_capital]
for r in ret_index:
    index_value.append(index_value[-1] * (1 + r))
index_value = pd.Series(index_value[1:], index=ret_index.index)  # align index with dates

In [8]:
# Equal-Weighted portfolio value (using RSP or average of stocks as proxy)
equal_value = [initial_capital]
for r in ret_equal_ref:
    equal_value.append(equal_value[-1] * (1 + r))
equal_value = pd.Series(equal_value[1:], index=ret_equal_ref.index)

In [9]:
# Momentum portfolio value
momentum_value = [initial_capital]
momentum_dates = stock_returns.index  # assuming this is a DatetimeIndex of monthly periods
# We will start the momentum strategy after 12 months (to have a lookback window)
lookback = 12  # 12-month lookback

In [11]:
lookback = 12  # 12-month lookback

pv = initial_capital
momentum_vals = []  # one value per date, aligned to momentum_dates

for i, _ in enumerate(momentum_dates):
    if i < lookback:
        # hold initial capital during warm-up (no trade)
        momentum_vals.append(pv)
    else:
        # rank on the last 12 months (i-12 ... i-1)
        window = stock_returns.iloc[i - lookback:i]
        cum_returns = (1 + window).prod() - 1
        top_stocks = cum_returns.nlargest(50).index
        # equal-weight return of selected basket for this month
        month_ret = stock_returns.iloc[i][top_stocks].mean()
        pv *= (1 + month_ret)
        momentum_vals.append(pv)

# perfectly aligned: same length as momentum_dates
momentum_series = pd.Series(momentum_vals, index=momentum_dates)

# if you want to display only the live period (after warm-up):
# momentum_series = momentum_series.iloc[lookback:]

Explanation: We calculated monthly returns for our data. For demonstration, we used SPY and RSP as proxies; in a full implementation, stock_returns would contain each of the 500 stocks’ monthly returns. We then simulated each strategy:
	•	For the Index, we multiplied the previous value by (1 + monthly return of SPY).
	•	For Equal Weight, we did the same using RSP’s returns (which implicitly represents the average performance of all stocks, rebalanced regularly). If we had all stock returns, we could instead take the average return each month (since equal weight means each stock contributes equally to performance each period).
	•	For Momentum, we looped through each month. After the first 12 months, at month i we computed each stock’s cumulative return over the prior 12 months (from i-12 to i-1). We picked the top 50 stocks with highest 12-month return and calculated the average of those stocks’ returns in month i (this is the return of the momentum portfolio for that month, assuming equal allocation among the winners). We then updated the portfolio value. We did not trade in the first year (just carried the initial capital) because we needed 12 months of data to rank stocks.

Note: The momentum strategy here is a simple long-only, equal-weighted momentum on top performers. More sophisticated approaches might exclude the most recent month’s return (to avoid short-term reversal, e.g. a 12-1 momentum strategy ￼) or use a different number of winners. Our goal is to illustrate the concept.

Also, we assumed zero trading costs. In a realistic scenario, the equal-weight and momentum strategies incur higher turnover (trading every month). To include transaction costs, one could subtract a small percentage (e.g. 0.1% per trade) from the portfolio value at each rebalance, or adjust the returns accordingly. We have left room in the code to integrate such costs if desired (for example, by multiplying (1 + month_ret) by (1 - cost_rate) for the momentum portfolio each rebalance).

Performance Metrics Calculation

With the portfolio equity curves computed for each strategy, we can now evaluate their performance. We will calculate:
	•	Cumulative Return (or ending portfolio value).
	•	Annualized Return (CAGR) – the compounded growth rate per year.
	•	Annualized Volatility – standard deviation of monthly returns, annualized (multiplied by √12, since we use monthly data).
	•	Sharpe Ratio – measures risk-adjusted return: (annual return minus risk-free rate) divided by annual volatility ￼. We will use the average 3-month T-bill rate as Rf. (If we retrieved rf_rate from FRED above, we’ll take the last value or average over the period; otherwise assume 2%.) A higher Sharpe ratio indicates better risk-adjusted performance ￼ ￼. For interpretability, we expect a Sharpe > 1.0 for good strategies in this bullish period (meaning the strategy’s returns exceeded the risk-free rate by more than one standard deviation of volatility).
	•	Maximum Drawdown – the largest peak-to-trough decline in portfolio value over the period ￼. This is a measure of downside risk: we calculate the running maximum of the portfolio value and see how deep the worst drop was from a peak. We will express drawdown as a percentage of the peak (e.g. a -20% drawdown means the portfolio lost 20% from its highest point before recovering).

In [18]:
# make sure these are Series (index: DatetimeIndex)
# index_value: equity curve for S&P 500 synthetic (Series)
# equal_value: equity curve for equal-weight portfolio (Series)
# momentum_series: equity curve for momentum strategy (Series)

# 1) Use min/max of each series’ index (not list.method .index!)
start_date = max(
    index_value.index.min(),
    equal_value.index.min(),
    momentum_series.index.min()
)
end_date = min(
    index_value.index.max(),
    equal_value.index.max(),
    momentum_series.index.max()
)

# 2) Build a common index and reindex
common_idx = (
    index_value.loc[start_date:end_date].index
    .intersection(equal_value.loc[start_date:end_date].index)
    .intersection(momentum_series.loc[start_date:end_date].index)
)

index_series = index_value.reindex(common_idx)
equal_series = equal_value.reindex(common_idx)
mom_series   = momentum_series.reindex(common_idx)

# Optional: drop any residual NaNs if your sources have missing months
index_series = index_series.dropna()
equal_series = equal_series.dropna()
mom_series   = mom_series.dropna()

In [23]:
# Calculate annualized return (CAGR)
years = (end_date - start_date).days / 365.25
cagr_index = (index_series[-1] / index_series[0]) ** (1/years) - 1
cagr_equal = (equal_series[-1] / equal_series[0]) ** (1/years) - 1
cagr_mom = (mom_series[-1] / mom_series[0]) ** (1/years) - 1

import numpy as np
import pandas as pd

# --- helpers ---
def cagr_from_curve(curve: pd.Series, periods_per_year=12) -> float:
    """CAGR from an equity curve with constant sampling frequency."""
    curve = curve.dropna()
    n_years = len(curve) / periods_per_year
    start = float(curve.iloc[0])
    end = float(curve.iloc[-1])
    return (end / start) ** (1.0 / n_years) - 1.0

def ann_vol_from_curve(curve: pd.Series, periods_per_year=12) -> float:
    """Annualized volatility from equity curve (via pct_change)."""
    rets = curve.pct_change().dropna()
    return float(rets.std() * np.sqrt(periods_per_year))

def sharpe_from_curve(curve: pd.Series, rf_ann=0.02, periods_per_year=12) -> float:
    """Annualized Sharpe using monthly excess returns."""
    rets = curve.pct_change().dropna()
    rf_m = rf_ann / periods_per_year
    ex_ret_m = rets - rf_m
    if ex_ret_m.std() == 0:
        return np.nan
    return float(ex_ret_m.mean() / ex_ret_m.std() * np.sqrt(periods_per_year))

def max_drawdown(curve: pd.Series) -> float:
    """Max drawdown of an equity curve (returns a negative number)."""
    curve = curve.dropna()
    roll_max = curve.cummax()
    dd = curve / roll_max - 1.0
    return float(dd.min())

# --- optional: risk-free from series rf_rate (e.g., 3M T-bill in %) ---
# If you have rf_rate as a Series in %, use its mean; else fallback to 2%.
try:
    rf_ann = float(rf_rate.mean().values) / 100.0
except Exception:
    rf_ann = 0.02

# --- metrics ---
cagr_index = cagr_from_curve(index_series)
cagr_equal = cagr_from_curve(equal_series)
cagr_mom   = cagr_from_curve(mom_series)

vol_index  = ann_vol_from_curve(index_series)
vol_equal  = ann_vol_from_curve(equal_series)
vol_mom    = ann_vol_from_curve(mom_series)

sharpe_index = sharpe_from_curve(index_series, rf_ann=rf_ann)
sharpe_equal = sharpe_from_curve(equal_series, rf_ann=rf_ann)
sharpe_mom   = sharpe_from_curve(mom_series,   rf_ann=rf_ann)

mdd_index = max_drawdown(index_series)
mdd_equal = max_drawdown(equal_series)
mdd_mom   = max_drawdown(mom_series)

# --- quick display ---
metrics = pd.DataFrame({
    "CAGR":   [cagr_index, cagr_equal, cagr_mom],
    "VolAnn": [vol_index,  vol_equal,  vol_mom],
    "Sharpe": [sharpe_index, sharpe_equal, sharpe_mom],
    "MaxDD":  [mdd_index,  mdd_equal,  mdd_mom],
}, index=["Index", "Equal-Weight", "Momentum"]).round(4)

metrics

/var/folders/d9/q7kdl2rj7k134djsnwv3q7xm0000gn/T/ipykernel_5748/1555037037.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cagr_index = (index_series[-1] / index_series[0]) ** (1/years) - 1
/var/folders/d9/q7kdl2rj7k134djsnwv3q7xm0000gn/T/ipykernel_5748/1555037037.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cagr_equal = (equal_series[-1] / equal_series[0]) ** (1/years) - 1
/var/folders/d9/q7kdl2rj7k134djsnwv3q7xm0000gn/T/ipykernel_5748/1555037037.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent 

,CAGR,VolAnn,Sharpe,MaxDD
Index,0.1429,0.1549,0.8180,-0.2397
Equal-Weight,0.1108,0.1711,0.5866,-0.2710
Momentum,0.1222,0.1580,0.6871,-0.2336
